In [0]:
from pyspark.sql import functions as F

catalog, schema = "workspace", "patient_notes"
base_path    = f"/Volumes/{catalog}/{schema}/raw_notes"
source_path  = f"{base_path}/incoming/"                  # Auto Loader also reads the day subfolders
checkpoint   = f"{base_path}/_checkpoints/bronze_notes"  # the bookmark
bronze_table = f"{catalog}.{schema}.bronze_notes"

In [0]:
raw_stream = (spark.readStream
    .format("cloudFiles")                                 # = Auto Loader
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint)      # where it remembers the columns
    .option("pathGlobFilter", "*.json")                   # skip _SUCCESS / _committed marker files
    .load(source_path))

bronze = (raw_stream
    .withColumn("source_file", F.col("_metadata.file_path"))   # which file each row came from
    .withColumn("ingested_at", F.current_timestamp()))          # when we loaded it

In [0]:
query = (bronze.writeStream
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)          # process all new files, then stop
    .toTable(bronze_table))

query.awaitTermination()                 # wait until it finishes
print("Rows in bronze:", spark.table(bronze_table).count())

In [0]:
df = spark.table(bronze_table)
display(df.limit(5))
display(df.groupBy("note_date").count())     # rows per day